# INTRODUCTION

The main objective of this project is to predict whether the home team will win. To achieve this, three separate datasets will be combined to create an analysis based on information available before each game. Each row in the final dataset will represent one game and will include pre-game information for both the home team and the away team. Using historical games that have already taken place, this information will be combined with the corresponding game outcomes. This will enable the development of a classification model that can predict whether the home team will win a specific future game.

In [114]:
#IMPORT LIBRARIES

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [115]:
# LOAD DATASET

df_games = pd.read_csv("games.csv")
df_teams = pd.read_csv("teams.csv")
df_ranking = pd.read_csv("ranking.csv")

In [116]:
df_games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2025-02-12,22400769,Final,1610612738,1610612759,2024,1610612738,116.0,0.462,0.667,...,25.0,56.0,1610612759,103.0,0.438,0.833,0.231,30.0,38.0,1
1,2025-02-12,22400770,Final,1610612753,1610612766,2024,1610612753,102.0,0.411,0.737,...,22.0,52.0,1610612766,86.0,0.395,0.933,0.308,21.0,43.0,1
2,2025-02-12,22400771,Final,1610612764,1610612754,2024,1610612764,130.0,0.485,0.813,...,29.0,46.0,1610612754,134.0,0.467,0.900,0.457,38.0,42.0,0
3,2025-02-12,22400772,Final,1610612751,1610612755,2024,1610612751,100.0,0.471,0.800,...,23.0,50.0,1610612755,96.0,0.432,0.818,0.242,25.0,38.0,1
4,2025-02-12,22400773,Final,1610612752,1610612737,2024,1610612752,149.0,0.574,0.811,...,28.0,39.0,1610612737,148.0,0.500,0.686,0.368,43.0,47.0,1


In [117]:
df_ranking.head()

,TEAM_ID,LEAGUE_ID,SEASON_ID,STANDINGSDATE,CONFERENCE,TEAM,G,W,L,W_PCT,HOME_RECORD,ROAD_RECORD,RETURNTOPLAY
0,1610612760,0,22024,2025-02-12,West,Oklahoma City,53,44,9,0.830,25-3,18-6,NaN
1,1610612763,0,22024,2025-02-12,West,Memphis,54,36,18,0.667,21-6,15-12,NaN
2,1610612743,0,22024,2025-02-12,West,Denver,55,36,19,0.655,19-8,17-11,NaN
3,1610612745,0,22024,2025-02-12,West,Houston,54,34,20,0.630,17-8,17-11,NaN
4,1610612747,0,22024,2025-02-12,West,L.A. Lakers,52,32,20,0.615,19-6,13-14,NaN


In [118]:
df_teams.head()

,LEAGUE_ID,TEAM_ID,MIN_YEAR,MAX_YEAR,ABBREVIATION,NICKNAME,YEARFOUNDED,CITY,ARENA,ARENACAPACITY,OWNER,GENERALMANAGER,HEADCOACH,DLEAGUEAFFILIATION
0,0,1610612737,1949,2019,ATL,Hawks,1949,Atlanta,State Farm Arena,18729.0,Tony Ressler,Travis Schlenk,Lloyd Pierce,Erie Bayhawks
1,0,1610612738,1946,2019,BOS,Celtics,1946,Boston,TD Garden,18624.0,Wyc Grousbeck,Danny Ainge,Brad Stevens,Maine Red Claws
2,0,1610612740,2002,2019,NOP,Pelicans,2002,New Orleans,Smoothie King Center,NaN,Tom Benson,Trajan Langdon,Alvin Gentry,No Affiliate
3,0,1610612741,1966,2019,CHI,Bulls,1966,Chicago,United Center,21711.0,Jerry Reinsdorf,Gar Forman,Jim Boylen,Windy City Bulls
4,0,1610612742,1980,2019,DAL,Mavericks,1980,Dallas,American Airlines Center,19200.0,Mark Cuban,Donnie Nelson,Rick Carlisle,Texas Legends


In [119]:
df_games.shape

(29755, 21)

In [120]:
df_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29755 entries, 0 to 29754
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     29755 non-null  object 
 1   GAME_ID           29755 non-null  int64  
 2   GAME_STATUS_TEXT  29755 non-null  object 
 3   HOME_TEAM_ID      29755 non-null  int64  
 4   VISITOR_TEAM_ID   29755 non-null  int64  
 5   SEASON            29755 non-null  int64  
 6   TEAM_ID_home      29755 non-null  int64  
 7   PTS_home          29656 non-null  float64
 8   FG_PCT_home       29656 non-null  float64
 9   FT_PCT_home       29656 non-null  float64
 10  FG3_PCT_home      29656 non-null  float64
 11  AST_home          29656 non-null  float64
 12  REB_home          29656 non-null  float64
 13  TEAM_ID_away      29755 non-null  int64  
 14  PTS_away          29656 non-null  float64
 15  FG_PCT_away       29656 non-null  float64
 16  FT_PCT_away       29656 non-null  float6

In [121]:
df_games.duplicated().sum()

0

In [122]:
df_games.isnull().sum()

GAME_DATE_EST        0
GAME_ID              0
GAME_STATUS_TEXT     0
HOME_TEAM_ID         0
VISITOR_TEAM_ID      0
SEASON               0
TEAM_ID_home         0
PTS_home            99
FG_PCT_home         99
FT_PCT_home         99
FG3_PCT_home        99
AST_home            99
REB_home            99
TEAM_ID_away         0
PTS_away            99
FG_PCT_away         99
FT_PCT_away         99
FG3_PCT_away        99
AST_away            99
REB_away            99
HOME_TEAM_WINS       0
dtype: int64

Twelve variables each contain 99 missing values. The next step is to check whether these values are missing in the same 99 rows, as this will help determine how to handle them.

The GAME_DATE_EST column contains dates but is currently stored as an object. Its data type will be addressed during the data preparation stage.

In [123]:
df_games[df_games["PTS_home"].isnull()]

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
22279,2003-10-24,10300116,Final,1610612753,1610612762,2003,1610612753,NaN,NaN,NaN,...,NaN,NaN,1610612762,NaN,NaN,NaN,NaN,NaN,NaN,0
22280,2003-10-24,10300108,Final,1610612737,1610612764,2003,1610612737,NaN,NaN,NaN,...,NaN,NaN,1610612764,NaN,NaN,NaN,NaN,NaN,NaN,0
22281,2003-10-24,10300109,Final,1610612738,1610612751,2003,1610612738,NaN,NaN,NaN,...,NaN,NaN,1610612751,NaN,NaN,NaN,NaN,NaN,NaN,0
22282,2003-10-24,10300113,Final,1610612759,1610612745,2003,1610612759,NaN,NaN,NaN,...,NaN,NaN,1610612745,NaN,NaN,NaN,NaN,NaN,NaN,0
22283,2003-10-24,10300112,Final,1610612749,1610612765,2003,1610612749,NaN,NaN,NaN,...,NaN,NaN,1610612765,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22373,2003-10-09,10300019,Final,1610612743,1610612756,2003,1610612743,NaN,NaN,NaN,...,NaN,NaN,1610612756,NaN,NaN,NaN,NaN,NaN,NaN,0
22374,2003-10-09,10300022,Final,1610612757,1610612758,2003,1610612757,NaN,NaN,NaN,...,NaN,NaN,1610612758,NaN,NaN,NaN,NaN,NaN,NaN,0
22375,2003-10-08,10300013,Final,1610612759,1610612763,2003,1610612759,NaN,NaN,NaN,...,NaN,NaN,1610612763,NaN,NaN,NaN,NaN,NaN,NaN,0
22382,2003-10-08,10300015,Final,1610612747,1610612744,2003,1610612747,NaN,NaN,NaN,...,NaN,NaN,1610612744,NaN,NaN,NaN,NaN,NaN,NaN,0


In [124]:
# Count missing values in each row
missing_per_row = df_games.isna().sum(axis=1)

# Count how many rows have at least one missing value
print((missing_per_row > 0).sum())

99


In [125]:
missing_per_row[missing_per_row > 0].value_counts()

12    99
Name: count, dtype: int64

In the dataset "df_games", there are 12 variables that contain missing values in the same 99 rows.

In [126]:
df_ranking.shape

(233832, 13)

In [127]:
df_ranking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233832 entries, 0 to 233831
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TEAM_ID        233832 non-null  int64  
 1   LEAGUE_ID      233832 non-null  int64  
 2   SEASON_ID      233832 non-null  int64  
 3   STANDINGSDATE  233832 non-null  object 
 4   CONFERENCE     233832 non-null  object 
 5   TEAM           233832 non-null  object 
 6   G              233832 non-null  int64  
 7   W              233832 non-null  int64  
 8   L              233832 non-null  int64  
 9   W_PCT          233832 non-null  float64
 10  HOME_RECORD    233832 non-null  object 
 11  ROAD_RECORD    233832 non-null  object 
 12  RETURNTOPLAY   3990 non-null    float64
dtypes: float64(2), int64(6), object(5)
memory usage: 23.2+ MB


In [128]:
df_ranking.duplicated().sum()

0

In [129]:
df_ranking.isnull().sum()

TEAM_ID               0
LEAGUE_ID             0
SEASON_ID             0
STANDINGSDATE         0
CONFERENCE            0
TEAM                  0
G                     0
W                     0
L                     0
W_PCT                 0
HOME_RECORD           0
ROAD_RECORD           0
RETURNTOPLAY     229842
dtype: int64

The STANDINGSDATE column contains dates but is currently stored as an object. Its data type will be addressed during the data preparation stage.

The RETURNTOPLAY column contains 229,842 missing values, accounting for approximately 98.3% of all rows. This column will be examined further to understand what it represents and determine how to handle the missing values.

In [130]:
df_ranking["RETURNTOPLAY"].value_counts(dropna=False)

RETURNTOPLAY
NaN    229842
1.0      2394
0.0      1596
Name: count, dtype: int64

The RETURNTOPLAY column has only 3,990 non-missing values. The recorded values are 0 and 1, which suggests that the variable is binary, although the meaning of these values is unclear. Given the high proportion of missing values and the column’s limited apparent relevance to predicting home-team wins, it is likely to be removed during data preparation. For this reason, further investigation of the column is a low priority.

In [131]:
# Quickly check the column to see if it contains the same information

df_ranking["LEAGUE_ID"].value_counts()

LEAGUE_ID
0    233832
Name: count, dtype: int64

In [132]:
df_ranking["CONFERENCE"].value_counts()

CONFERENCE
East    117103
West    116729
Name: count, dtype: int64

The last two columns examined were LEAGUE_ID and CONFERENCE, both of which were flagged for closer inspection. To assess their relevance, the main question was “How could this information help predict whether the home team will win?”

LEAGUE_ID has the same value in every row, so it cannot help the model distinguish between games. CONFERENCE has two categories, East and West, which appear to be fairly balanced. These categories reflect how the league groups its teams geographically. A balanced distribution alone provides little evidence that conference membership helps predict home-team wins.

Based on these observations, LEAGUE_ID can be removed during data preparation, and CONFERENCE is also a candidate for removal.

In [133]:
df_teams.head()

,LEAGUE_ID,TEAM_ID,MIN_YEAR,MAX_YEAR,ABBREVIATION,NICKNAME,YEARFOUNDED,CITY,ARENA,ARENACAPACITY,OWNER,GENERALMANAGER,HEADCOACH,DLEAGUEAFFILIATION
0,0,1610612737,1949,2019,ATL,Hawks,1949,Atlanta,State Farm Arena,18729.0,Tony Ressler,Travis Schlenk,Lloyd Pierce,Erie Bayhawks
1,0,1610612738,1946,2019,BOS,Celtics,1946,Boston,TD Garden,18624.0,Wyc Grousbeck,Danny Ainge,Brad Stevens,Maine Red Claws
2,0,1610612740,2002,2019,NOP,Pelicans,2002,New Orleans,Smoothie King Center,NaN,Tom Benson,Trajan Langdon,Alvin Gentry,No Affiliate
3,0,1610612741,1966,2019,CHI,Bulls,1966,Chicago,United Center,21711.0,Jerry Reinsdorf,Gar Forman,Jim Boylen,Windy City Bulls
4,0,1610612742,1980,2019,DAL,Mavericks,1980,Dallas,American Airlines Center,19200.0,Mark Cuban,Donnie Nelson,Rick Carlisle,Texas Legends


In [134]:
df_teams.shape

(30, 14)

In [135]:
df_teams.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   LEAGUE_ID           30 non-null     int64  
 1   TEAM_ID             30 non-null     int64  
 2   MIN_YEAR            30 non-null     int64  
 3   MAX_YEAR            30 non-null     int64  
 4   ABBREVIATION        30 non-null     object 
 5   NICKNAME            30 non-null     object 
 6   YEARFOUNDED         30 non-null     int64  
 7   CITY                30 non-null     object 
 8   ARENA               30 non-null     object 
 9   ARENACAPACITY       26 non-null     float64
 10  OWNER               30 non-null     object 
 11  GENERALMANAGER      30 non-null     object 
 12  HEADCOACH           30 non-null     object 
 13  DLEAGUEAFFILIATION  30 non-null     object 
dtypes: float64(1), int64(5), object(8)
memory usage: 3.4+ KB


In [136]:
df_teams.isnull().sum()

LEAGUE_ID             0
TEAM_ID               0
MIN_YEAR              0
MAX_YEAR              0
ABBREVIATION          0
NICKNAME              0
YEARFOUNDED           0
CITY                  0
ARENA                 0
ARENACAPACITY         4
OWNER                 0
GENERALMANAGER        0
HEADCOACH             0
DLEAGUEAFFILIATION    0
dtype: int64

In [137]:
df_teams.duplicated().sum()

0

In [138]:
df_teams["TEAM_ID"].nunique()

30

In [139]:
df_teams["ARENACAPACITY"].value_counts(dropna=False)

ARENACAPACITY
NaN        4
19060.0    2
17500.0    2
18729.0    1
19980.0    1
20562.0    1
19026.0    1
21000.0    1
20647.0    1
18119.0    1
20148.0    1
19800.0    1
19163.0    1
18694.0    1
0.0        1
18345.0    1
18624.0    1
19763.0    1
19356.0    1
19600.0    1
18104.0    1
19099.0    1
19200.0    1
21711.0    1
19596.0    1
Name: count, dtype: int64

In [140]:
df_teams["ARENACAPACITY"].nunique(dropna=False)

25

The df_teams dataset was analysed because it will serve as a reference table for linking team names to records in the other datasets. It contains 30 rows and 30 unique TEAM_ID values, giving each team ID exactly one matching record in df_teams. This allows the team name, stored in the NICKNAME column, to be retrieved using TEAM_ID without creating duplicate rows due to multiple matches in the reference table.

The ARENACAPACITY column contains four missing values and one recorded value of zero. Arena capacity will be excluded from the model because its relevance to predicting home-team wins is unclear.

# Characterisation Summary
The three datasets were examined to understand their structure, review the types of variables they contain, and identify missing values and potential inconsistencies. The games dataset contains 29,755 rows and 21 columns, the ranking dataset contains 233,832 rows and 13 columns, and the teams dataset contains 30 rows and 14 columns. The datasets will be used together to address the project’s central question: can we predict whether the home team will win a game using only information available before it starts? The target variable, HOME_TEAM_WINS, indicates whether the home team won.

The initial review showed that 12 game-statistics variables contain 99 missing values each, occurring in the same 99 rows. These variables are particularly relevant because some of them may later be used to create pre-game features based on previous games. Other missing values and columns with limited relevance were also identified. These findings will be used in the Data Preparation stage to decide how missing values should be handled, which variables should be kept, and which new features should be created for the model.

# Data Preparation

Correct data types → Data Cleaning (remove or set aside irrelevant columns and handle the 99 missing values) → Create new pre-game features, such as home_avg_pts_last5, away_avg_pts_last5, and home_recent_win_pct_last5, and add the latest ranking information available before each game (which will help answer the project’s central question and predict the target).

In [141]:
# I used .copy() to create an independent cleaned dataframe and avoid modifying a view of the original dataset

df_games_clean = df_games.copy()
df_ranking_clean = df_ranking.copy()
df_teams_clean = df_teams.copy()

In [142]:
df_ranking_clean["STANDINGSDATE"] = pd.to_datetime(df_ranking_clean["STANDINGSDATE"])

In [143]:
df_ranking_clean["STANDINGSDATE"].dtype

dtype('<M8[ns]')

In [144]:
df_games_clean["GAME_DATE_EST"] = pd.to_datetime(df_games_clean["GAME_DATE_EST"])

In [145]:
df_games_clean["GAME_DATE_EST"].dtype

dtype('<M8[ns]')

The analysis will use team statistics that were available before each game. As a first step, the STANDINGSDATE and GAME_DATE_EST columns will be converted to a datetime data type so that the dates can be sorted and compared chronologically. This will make it possible to identify the most recent standings dated before each game for both the home and away teams.

In [146]:
# Remove columns with lower target adherence

df_ranking_clean = df_ranking_clean.drop(columns=['LEAGUE_ID', 'CONFERENCE', 'RETURNTOPLAY'])

In [147]:
df_ranking_clean.columns

Index(['TEAM_ID', 'SEASON_ID', 'STANDINGSDATE', 'TEAM', 'G', 'W', 'L', 'W_PCT',
       'HOME_RECORD', 'ROAD_RECORD'],
      dtype='object')

In [153]:
df_teams_clean = df_teams_clean.drop(columns=['ARENACAPACITY'])

In [154]:
# understanding 99 missings values

missing_games = df_games_clean[df_games_clean["PTS_home"].isnull()]
missing_games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
22279,2003-10-24,10300116,Final,1610612753,1610612762,2003,1610612753,NaN,NaN,NaN,...,NaN,NaN,1610612762,NaN,NaN,NaN,NaN,NaN,NaN,0
22280,2003-10-24,10300108,Final,1610612737,1610612764,2003,1610612737,NaN,NaN,NaN,...,NaN,NaN,1610612764,NaN,NaN,NaN,NaN,NaN,NaN,0
22281,2003-10-24,10300109,Final,1610612738,1610612751,2003,1610612738,NaN,NaN,NaN,...,NaN,NaN,1610612751,NaN,NaN,NaN,NaN,NaN,NaN,0
22282,2003-10-24,10300113,Final,1610612759,1610612745,2003,1610612759,NaN,NaN,NaN,...,NaN,NaN,1610612745,NaN,NaN,NaN,NaN,NaN,NaN,0
22283,2003-10-24,10300112,Final,1610612749,1610612765,2003,1610612749,NaN,NaN,NaN,...,NaN,NaN,1610612765,NaN,NaN,NaN,NaN,NaN,NaN,0


In [155]:
missing_games.shape

(99, 21)

In [156]:
missing_games["GAME_STATUS_TEXT"].value_counts()

GAME_STATUS_TEXT
Final    99
Name: count, dtype: int64

In [157]:
missing_games["SEASON"].value_counts().sort_index()

SEASON
2003    99
Name: count, dtype: int64

In [160]:
# checking where these missings vlues are

games_2003 = df_games_clean[df_games_clean["SEASON"] == 2003]

total_games_2003 = len(games_2003)
missing_percentage_2003 = (len(missing_games) / total_games_2003) * 100

print(total_games_2003)
print(missing_percentage_2003)


1385
7.148014440433213


In [161]:
missing_games.sort_values("GAME_DATE_EST")[
    ["GAME_DATE_EST", "GAME_ID"]
]

,GAME_DATE_EST,GAME_ID
22383,2003-10-07,10300006
22382,2003-10-08,10300015
22375,2003-10-08,10300013
22371,2003-10-09,10300020
22374,2003-10-09,10300022
...,...,...
22283,2003-10-24,10300112
22282,2003-10-24,10300113
22281,2003-10-24,10300109
22280,2003-10-24,10300108


In [162]:
missing_games["GAME_DATE_EST"].min()

Timestamp('2003-10-07 00:00:00')

In [163]:
missing_games["GAME_DATE_EST"].max()

Timestamp('2003-10-24 00:00:00')

In [164]:
missing_games["GAME_DATE_EST"].dt.to_period("M").value_counts().sort_index()

GAME_DATE_EST
2003-10    99
Freq: M, Name: count, dtype: int64

In [170]:
# first and last date with complete stastical

complete_games_2003 = games_2003[games_2003["PTS_home"].notnull()]
complete_games_2003["GAME_DATE_EST"].min()

Timestamp('2003-10-05 00:00:00')

In [169]:
complete_games_2003["GAME_DATE_EST"].max()

Timestamp('2004-06-15 00:00:00')

In [171]:
games_october_2003 = games_2003[
    (games_2003["GAME_DATE_EST"] >= "2003-10-07") &
    (games_2003["GAME_DATE_EST"] <= "2003-10-24")
]

In [172]:
games_october_2003["PTS_home"].isnull().value_counts()

PTS_home
True     99
False    13
Name: count, dtype: int64

I know that the matches from 7 to 24 October 2003 have missing values. However, I would also like to know whether there are any complete matches within that period. This is important so that I can begin to formulate my approach to handling the missing data. It can therefore be seen that there are 99 matches with missing PTS_Home values and 13 matches with complete PTS_Home values.

In [173]:
games_october_2003.groupby("GAME_DATE_EST")["PTS_home"].agg(
    total_games="size",
    missing_games=lambda x: x.isnull().sum()
)

,total_games,missing_games
GAME_DATE_EST,,
2003-10-07,8,1
2003-10-08,8,2
2003-10-09,4,4
2003-10-10,5,5
2003-10-11,9,9
2003-10-12,3,3
2003-10-13,2,2
2003-10-14,10,10
2003-10-15,4,4


On the 7th, out of the 8 matches, 1 has a missing value. On the 8th, however, there are only 2 missing values. However, from 9 October 2003 onwards, all matches have missing values. It is therefore already apparent that this does not appear to be a random problem scattered throughout the dataset. There is a strong temporal concentration between 9 and 24 October 2003.

In [181]:
home_counts = missing_games["HOME_TEAM_ID"].value_counts()
away_counts = missing_games["VISITOR_TEAM_ID"].value_counts()

team_counts = pd.concat([home_counts, away_counts], axis=1)
team_counts.columns = ["Home", "Away"]

team_counts

,Home,Away
1610612747,8,NaN
1610612759,5,3.0
1610612765,5,1.0
1610612746,5,2.0
1610612753,4,2.0
1610612752,4,3.0
1610612751,4,3.0
1610612757,4,3.0
1610612745,4,2.0
1610612741,4,3.0


This check was carried out to identify which teams appear among these 99 missing values. It is also important to note that this NaN does not represent a new missing value in the dataset. It simply means that this team appeared eight times as HOME_TEAM_ID within the 99 matches, but not once as VISITOR_TEAM_ID in this subset.

In [182]:
team_counts = team_counts.fillna(0)

In [183]:
team_counts["Total"] = team_counts["Home"] + team_counts["Away"]

In [185]:
team_counts = team_counts.astype(int)
team_counts

,Home,Away,Total
1610612747,8,0,8
1610612759,5,3,8
1610612765,5,1,6
1610612746,5,2,7
1610612753,4,2,6
1610612752,4,3,7
1610612751,4,3,7
1610612757,4,3,7
1610612745,4,2,6
1610612741,4,3,7


In [188]:
# The number of matches each team played in this season
# compared with the number of matches for which that team has missing values, 
# and the percentage of those matches that are among the 99 matches with missing statistics

In [189]:
home_total_2003 = games_2003["HOME_TEAM_ID"].value_counts()
away_total_2003 = games_2003["VISITOR_TEAM_ID"].value_counts()

In [191]:
total_games_by_team_2003 = home_total_2003.add(
    away_total_2003,
    fill_value=0
).astype(int)
total_games_by_team_2003

1610612737     90
1610612738     94
1610612739     90
1610612740     97
1610612741     90
1610612742     95
1610612743     95
1610612744     89
1610612745     95
1610612746     90
1610612747    112
1610612748    102
1610612749     95
1610612750    108
1610612751    101
1610612752     94
1610612753     90
1610612754    106
1610612755     90
1610612756     90
1610612757     90
1610612758    102
1610612759    100
1610612760     90
1610612761     89
1610612762     90
1610612763     93
1610612764     90
1610612765    113
Name: count, dtype: int64

In [192]:
team_impact_2003 = pd.DataFrame({
    "Total_Games": total_games_by_team_2003,
    "Missing_Games": team_counts["Total"]
})

team_impact_2003["Missing_Percentage"] = (
    team_impact_2003["Missing_Games"] /
    team_impact_2003["Total_Games"]
) * 100
team_impact_2003

In [196]:
team_impact_2003["Missing_Percentage"].min()

5.3097345132743365

In [197]:
team_impact_2003["Missing_Percentage"].max()

8.421052631578947

The analysis showed that missing values affect a significant proportion of each team’s games. Since these statistics will be used to create new features, missing values may affect how those features are calculated. It is therefore important to investigate the affected games before deciding how to handle the missing data. The affected games are concentrated at the start of the 2003 season. Checking whether they were played in the pre-season or the regular season could help determine how to handle them. If they were pre-season games and fall outside the project's scope, they could be excluded. If the games are retained and mean imputation is needed, the mean should be calculated from comparable games in the same category. Pre-season and regular-season data should be combined for imputation only after assessing whether their statistics are comparable.

In [198]:
missing_games["GAME_ID"].astype(str).str.zfill(10).str[:3].value_counts()

GAME_ID
001    99
Name: count, dtype: int64

In [199]:
games_2003["GAME_ID"].astype(str).str.zfill(10).str[:3].value_counts()

GAME_ID
002    1189
001     114
004      82
Name: count, dtype: int64

In [200]:
games_2003.assign(
    GAME_TYPE_CODE=games_2003["GAME_ID"].astype(str).str.zfill(10).str[:3]
).groupby("GAME_TYPE_CODE")["GAME_DATE_EST"].agg(["min", "max", "count"])

,min,max,count
GAME_TYPE_CODE,,,
001,2003-10-05,2003-10-24,114
002,2003-10-28,2004-04-14,1189
004,2004-04-17,2004-06-15,82


According to the NBA Data project documentation, the first three digits of the GAME_ID identify the game type, where 001represents preseason, 002 regular season, and 004 playoffs (Llimllib, 2026). It was therefore possible to observe that all 99 missing values are associated with games with the prefix 001, indicating that they belong to the preseason. The dates were also checked by comparing games with the prefix 001 with the other identified types: 002, corresponding to the regular season, and 004, corresponding to the playoffs. Games with the prefix 001 occur before the other periods, which is also consistent with the preseason classification. In total, 114 preseason games were identified in the 2003 season, of which 99 contain missing values, representing approximately 87% of this category. Even after these analyses, it is not yet possible to determine the most appropriate treatment for these missing values. Therefore, further investigation is needed before deciding whether to remove, replace, or apply another treatment strategy.

In [201]:
df_games_clean["GAME_TYPE_CODE"] = (
    df_games_clean["GAME_ID"]
    .astype(str)
    .str.zfill(10)
    .str[:3]
)

In [202]:
preseason_games = df_games_clean[
    df_games_clean["GAME_TYPE_CODE"] == "001"
]

In [203]:
preseason_by_season = preseason_games.groupby("SEASON")["PTS_home"].agg(
    total_games="size",
    missing_games=lambda x: x.isnull().sum()
)

preseason_by_season

,total_games,missing_games
SEASON,,
2003,114,99
2004,48,0
2005,113,0
2006,110,0
2007,95,0
2008,110,0
2009,112,0
2010,111,0
2011,30,0


2003 → 114 preseason games → 99 missing

2004–2024 → preseason games → 0 missing

All of this analysis was important because historical game data will be used to create new features based on teams’ recent performance, such as averages from the last five games. Consequently, preseason, regular season and playoff games can influence the construction of these features and, therefore, the model used to answer the study’s main question: can we predict whether the home team will win a game using only information available before it starts?

The investigation showed that the 99 missing values are concentrated in the 2003 preseason. However, at this stage, no decision will be made yet about whether to remove these records or replace their missing values. No further specific analysis of these 99 records will be carried out for now. The analysis will next focus on the remaining information required for feature engineering, particularly the information obtained from ranking.csv, such as home_w_pct, away_w_pct, home_home_win_pct and away_road_win_pct. Once the availability and behaviour of this information have also been investigated, it will be possible to make a more informed decision about how to handle the missing values.